# 🤖 Chatbot con Fallback Automático (OpenAI → Anthropic → Google)

**Reto proyecto — Desarrollo de Soluciones IA**

Chatbot en Python con un sistema de *fallback* en cascada entre tres proveedores de IA. Por defecto usa **OpenAI**; si falla pasa a **Anthropic Claude**; si este también falla pasa a **Google Gemini**; y si ninguno responde, devuelve una **respuesta preconfigurada**. El historial completo de la conversación se mantiene **en memoria** y se conserva entre cambios de proveedor.

### Mapeo de la estructura de archivos pedida → celdas de este notebook

```
├── main.py                      → Celda «CLI» (run_cli)
├── providers/
│   ├── __init__.py              → (implícito)
│   ├── openai_provider.py       → Celda «Proveedor OpenAI»
│   ├── anthropic_provider.py    → Celda «Proveedor Anthropic»
│   └── gemini_provider.py       → Celda «Proveedor Google Gemini»
├── core/
│   ├── __init__.py              → (implícito)
│   ├── chatbot.py               → Celda «Lógica del chatbot + fallback»
│   └── conversation.py          → Celda «Gestión del historial»
├── requirements.txt             → Celda de dependencias
├── .env                         → Celda de configuración
└── README.md                    → Este encabezado
```

> **Nota sobre las API KEY:** no es necesario incluir las claves al entregar la solución. > Se cargan desde un archivo `.env` local mediante `python-dotenv`.

## 1. Dependencias (`requirements.txt`)

```text
openai
anthropic
google-genai
python-dotenv
```

Ejecuta la siguiente celda una sola vez para instalarlas.

In [1]:
# Instalación de los SDK oficiales (ejecutar una vez)
%pip install -q openai anthropic google-genai python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Configuración (`.env`)

Crea un archivo `.env` en la misma carpeta que este notebook con tus claves. **No subas este archivo** a tu entrega; mantenlo en tu entorno local.

```dotenv
OPENAI_API_KEY=sk-...
ANTHROPIC_API_KEY=sk-ant-...
GEMINI_API_KEY=AIza...
OPENAI_MODEL=gpt-5-mini
ANTHROPIC_MODEL=claude-haiku-4-5-20251001
GEMINI_MODEL=gemini-2.5-flash
SYSTEM_PROMPT=Eres un asistente útil y conciso. Respondes en español.
FALLBACK_MESSAGE=Lo siento, ahora mismo no puedo procesar tu mensaje. Intenta de nuevo más tarde.
```

La siguiente celda carga esas variables de entorno y permite usar toda la configuración desde `.env`.

In [2]:
import os
from dotenv import load_dotenv

# Carga las variables de entorno definidas en .env
load_dotenv()

# Comprobación informativa (no muestra las claves)
for var in ("OPENAI_API_KEY", "ANTHROPIC_API_KEY", "GEMINI_API_KEY"):
    estado = "✅ configurada" if os.getenv(var) else "❌ no encontrada"
    print(f"{var}: {estado}")

OPENAI_API_KEY: ✅ configurada
ANTHROPIC_API_KEY: ✅ configurada
GEMINI_API_KEY: ✅ configurada


## 3. Gestión del historial (`core/conversation.py`)

Clase `Conversation` que guarda el historial completo usuario↔asistente **en memoria** (sin base de datos externa). Mantiene un formato canónico de mensajes `{"role": "user"|"assistant", "content": str}` que cada proveedor traduce a su propio formato. El *system prompt* se almacena por separado porque cada SDK lo trata de forma distinta.

In [3]:
class Conversation:
    """Mantiene el historial de la conversación en memoria."""

    def __init__(self, system_prompt: str | None = None):
        self.system_prompt = system_prompt
        self._messages: list[dict] = []

    def add_user_message(self, content: str) -> None:
        self._messages.append({"role": "user", "content": content})

    def add_assistant_message(self, content: str) -> None:
        self._messages.append({"role": "assistant", "content": content})

    @property
    def messages(self) -> list[dict]:
        """Copia del historial en formato canónico (user/assistant)."""
        return list(self._messages)

    def clear(self) -> None:
        self._messages.clear()

    def __len__(self) -> int:
        return len(self._messages)

    def __repr__(self) -> str:
        return f"Conversation({len(self._messages)} mensajes)"

## 4. Interfaz común de proveedores

Todos los proveedores comparten la misma interfaz: un método `generate(conversation)` que **transmite la respuesta en streaming** por pantalla y devuelve el texto completo, o lanza una excepción si falla. Esto permite que la lógica de fallback los trate de forma intercambiable.

Los clientes de cada SDK se crean de forma **perezosa** (*lazy*): así, si falta una API KEY, el proveedor simplemente falla al generar y se activa el fallback, en lugar de romper el notebook al definir las clases.

In [4]:
class BaseProvider:
    """Interfaz común a todos los proveedores."""
    name: str = "base"

    def generate(self, conversation: "Conversation") -> str:
        raise NotImplementedError

## 5. Proveedor OpenAI (`providers/openai_provider.py`)

* Cliente `OpenAI` del SDK oficial.
* **API Responses** (`client.responses.create`), *no* Chat Completions.
* Modelo configurable (por defecto `gpt-5-mini`).
* **Streaming** activado: se procesan los eventos `response.output_text.delta`.
* El *system prompt* se pasa mediante el parámetro `instructions`.

In [5]:
class OpenAIProvider(BaseProvider):
    name = "OpenAI"

    def __init__(self, model: str = "gpt-5-mini", api_key: str | None = None):
        self.model = model
        self._api_key = api_key
        self._client = None

    @property
    def client(self):
        if self._client is None:
            from openai import OpenAI
            self._client = OpenAI(
                api_key=self._api_key or os.getenv("OPENAI_API_KEY"),
                timeout=30.0,      # falla rápido si no responde
                max_retries=1,
            )
        return self._client

    def generate(self, conversation: "Conversation") -> str:
        kwargs = {
            "model": self.model,
            "input": conversation.messages,   # historial en formato role/content
            "stream": True,                    # modo streaming
        }
        if conversation.system_prompt:
            kwargs["instructions"] = conversation.system_prompt

        # API Responses en streaming (lanza excepción si la clave o la red fallan)
        stream = self.client.responses.create(**kwargs)

        partes: list[str] = []
        for event in stream:
            # En la API Responses el texto llega en eventos de tipo
            # "response.output_text.delta", con el fragmento en event.delta.
            if event.type == "response.output_text.delta":
                print(event.delta, end="", flush=True)
                partes.append(event.delta)
        print()
        return "".join(partes)


## 6. Proveedor Anthropic (`providers/anthropic_provider.py`)

* Cliente `anthropic.Anthropic` del SDK oficial.
* Modelo configurable (por defecto `claude-haiku-4-5-20251001`).
* **Streaming** con `client.messages.stream(...)` e iterando sobre `stream.text_stream`.
* En Anthropic el *system prompt* va en el parámetro `system` (fuera de `messages`) y `max_tokens` es obligatorio. El historial `user/assistant` es directamente compatible.

In [6]:
class AnthropicProvider(BaseProvider):
    name = "Anthropic"

    def __init__(self, model: str = "claude-haiku-4-5-20251001",
                 api_key: str | None = None, max_tokens: int = 1024):
        self.model = model
        self.max_tokens = max_tokens
        self._api_key = api_key
        self._client = None

    @property
    def client(self):
        if self._client is None:
            import anthropic
            self._client = anthropic.Anthropic(
                api_key=self._api_key or os.getenv("ANTHROPIC_API_KEY"),
                timeout=30.0,      # falla rápido si no responde
                max_retries=1,
            )
        return self._client

    def generate(self, conversation: "Conversation") -> str:
        kwargs = {
            "model": self.model,
            "max_tokens": self.max_tokens,
            "messages": conversation.messages,  # user/assistant compatible
        }
        if conversation.system_prompt:
            kwargs["system"] = conversation.system_prompt

        partes: list[str] = []
        # Streaming: text_stream entrega el texto por fragmentos a medida que llega
        with self.client.messages.stream(**kwargs) as stream:
            for texto in stream.text_stream:
                print(texto, end="", flush=True)
                partes.append(texto)
        print()
        return "".join(partes)


## 7. Proveedor Google Gemini (`providers/gemini_provider.py`)

* SDK `google-genai` (`from google import genai`).
* Modelo configurable (por defecto `gemini-2.5-flash`).
* **Streaming** con `client.models.generate_content_stream(...)`.
* Google usa otro formato: roles `user`/`model` (el asistente es `model`) y `parts`. El *system prompt* se pasa en `GenerateContentConfig(system_instruction=...)`. Se incluye un conversor del formato canónico al de Gemini.

In [7]:
class GeminiProvider(BaseProvider):
    name = "Google Gemini"

    def __init__(self, model: str = "gemini-2.5-flash", api_key: str | None = None):
        self.model = model
        self._api_key = api_key
        self._client = None

    @property
    def client(self):
        if self._client is None:
            from google import genai
            from google.genai import types
            self._client = genai.Client(
                api_key=self._api_key or os.getenv("GEMINI_API_KEY"),
                http_options=types.HttpOptions(timeout=30_000),  # 30 s (en ms)
            )
        return self._client

    @staticmethod
    def _to_contents(conversation: Conversation) -> list[dict]:
        """Convierte el historial canónico al formato de Gemini (user/model + parts)."""
        contents = []
        for m in conversation.messages:
            role = "model" if m["role"] == "assistant" else "user"
            contents.append({"role": role, "parts": [{"text": m["content"]}]})
        return contents

    def generate(self, conversation: Conversation) -> str:
        from google.genai import types

        config = None
        if conversation.system_prompt:
            config = types.GenerateContentConfig(
                system_instruction=conversation.system_prompt
            )

        stream = self.client.models.generate_content_stream(  # modo streaming
            model=self.model,
            contents=self._to_contents(conversation),
            config=config,
        )

        partes: list[str] = []
        for chunk in stream:
            if chunk.text:
                print(chunk.text, end="", flush=True)
                partes.append(chunk.text)
        print()
        return "".join(partes)

## 8. Lógica del chatbot y fallback (`core/chatbot.py`)

La clase `Chatbot` recibe la lista ordenada de proveedores y orquesta la **cascada**:

1. Añade el mensaje del usuario al historial.
2. Intenta cada proveedor **en orden de prioridad** (OpenAI → Anthropic → Google).
3. Captura cualquier excepción (errores de conectividad, límites de API, claves ausentes, respuestas vacías…), **notifica el cambio de proveedor** y pasa al siguiente.
4. Si los tres fallan, responde con un **mensaje preconfigurado**.
5. En todos los casos guarda la respuesta del asistente para **mantener la continuidad** del historial entre cambios de proveedor.

In [8]:
class Chatbot:
    """Orquesta la conversación con fallback en cascada entre proveedores."""

    DEFAULT_FALLBACK = (
        "Lo siento, ahora mismo no puedo procesar tu mensaje porque ninguno de los "
        "proveedores de IA está disponible. Por favor, inténtalo de nuevo más tarde."
    )

    def __init__(self, providers: list[BaseProvider],
                 system_prompt: str | None = None,
                 fallback_message: str | None = None):
        self.providers = providers
        self.conversation = Conversation(system_prompt=system_prompt)
        self.fallback_message = fallback_message or self.DEFAULT_FALLBACK

    def send(self, user_input: str) -> str:
        """Envía un mensaje del usuario y devuelve la respuesta del asistente."""
        self.conversation.add_user_message(user_input)

        for i, provider in enumerate(self.providers):
            try:
                print(f"\n[Asistente · {provider.name}]: ", end="", flush=True)
                respuesta = provider.generate(self.conversation)

                if not respuesta.strip():
                    raise RuntimeError("respuesta vacía del proveedor")

                # Éxito: guardamos en el historial y devolvemos
                self.conversation.add_assistant_message(respuesta)
                return respuesta

            except Exception as e:
                # Detección y notificación del fallo
                print(f"\n⚠️  {provider.name} no está disponible "
                      f"({type(e).__name__}: {e}).")
                if i + 1 < len(self.providers):
                    print(f"➡️  Cambiando a {self.providers[i + 1].name}…")
                continue

        # Todos los proveedores fallaron: respuesta preconfigurada
        print("\n❌ Ningún proveedor respondió. Usando respuesta preconfigurada.")
        print(f"[Asistente · fallback]: {self.fallback_message}")
        self.conversation.add_assistant_message(self.fallback_message)
        return self.fallback_message

## 9. Interfaz CLI (`main.py`)

Bucle conversacional interactivo. Escribe tus mensajes y pulsa Enter. Para terminar de forma limpia escribe **`/salir`** (también valen `/exit` o `/quit`, o `Ctrl-C`).

En un notebook, la función `input()` abre un campo de texto sobre la celda en ejecución.

In [9]:
def run_cli():
    """Bucle principal de la interfaz de línea de comandos."""
    load_dotenv()

    # Configuración desde .env con valores por defecto cuando no están definidos
    openai_model = os.getenv("OPENAI_MODEL", "gpt-5-mini")
    anthropic_model = os.getenv("ANTHROPIC_MODEL", "claude-haiku-4-5-20251001")
    gemini_model = os.getenv("GEMINI_MODEL", "gemini-2.5-flash")
    system_prompt = os.getenv(
        "SYSTEM_PROMPT",
        "Eres un asistente útil y conciso. Respondes en español.",
    )
    fallback_message = os.getenv("FALLBACK_MESSAGE")

    # Orden de prioridad del fallback: OpenAI → Anthropic → Google
    providers = [
        OpenAIProvider(model=openai_model),
        AnthropicProvider(model=anthropic_model),
        GeminiProvider(model=gemini_model),
    ]

    bot = Chatbot(
        providers,
        system_prompt=system_prompt,
        fallback_message=fallback_message,
    )

    print("=" * 60)
    print("🤖 Chatbot con fallback (OpenAI → Anthropic → Google)")
    print("Escribe '/salir' para terminar.")
    print("=" * 60)

    while True:
        try:
            user_input = input("\n[Tú]: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\n👋 ¡Hasta luego!")
            break

        if not user_input:
            continue

        if user_input.lower() in ("/salir", "/exit", "/quit"):
            print("👋 ¡Hasta luego!")
            break

        bot.send(user_input)

### ▶️ Iniciar la conversación

Ejecuta la celda siguiente para lanzar el chatbot.

### ▶️ Opción A — Conversar sin `input()` (recomendado en notebooks)

Si el cuadro de texto de `input()` no te aparece o se queda colgado, usa este método: creas el bot **una sola vez** y luego le hablas pasando el mensaje como argumento de `bot.send(...)`. El historial se mantiene porque reutilizas el mismo objeto `bot`.

**Importante:** antes de ejecutar estas celdas, ejecuta de arriba abajo las celdas de `Conversation`, los tres proveedores y `Chatbot` (o reinicia el kernel y ejecútalo todo).

In [10]:
# Crear el bot UNA sola vez (vuelve a ejecutar esta celda para reiniciar la conversación)
load_dotenv()

bot = Chatbot(
    [
        OpenAIProvider(model="gpt-5-mini"),
        AnthropicProvider(model="claude-haiku-4-5-20251001"),
        GeminiProvider(model="gemini-2.5-flash"),
    ],
    system_prompt="Eres un asistente útil y conciso. Respondes en español.",
)
print("Bot listo. Usa bot.send(\"tu mensaje\") en la siguiente celda.")

Bot listo. Usa bot.send("tu mensaje") en la siguiente celda.


In [11]:
# Cambia el texto y ejecuta esta celda tantas veces como quieras: el historial se conserva
bot.send("Hola, ¿cómo estás?")


[Asistente · OpenAI]: 
⚠️  OpenAI no está disponible (APIError: You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.).
➡️  Cambiando a Anthropic…

[Asistente · Anthropic]: 
⚠️  Anthropic no está disponible (BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.'}, 'request_id': 'req_011CbqV41Jczy8p8jYqAXtQB'}).
➡️  Cambiando a Google Gemini…

[Asistente · Google Gemini]: Estoy bien, gracias. ¿Y tú?


'Estoy bien, gracias. ¿Y tú?'

In [12]:
# (Opcional) Ver el historial completo en memoria
for m in bot.conversation.messages:
    print(f"{m['role']}: {m['content']}")

user: Hola, ¿cómo estás?
assistant: Estoy bien, gracias. ¿Y tú?


### ▶️ Opción B — CLI interactiva con `run_cli()`

Usa el bucle con `input()` y salida con `/salir`. Funciona en una terminal real (`python main.py`). En notebooks, el cuadro de texto aparece debajo de la celda (Jupyter/Colab) o arriba en la paleta de comandos (VS Code).

> Si esta celda se queda en `[*]` mucho tiempo, púlsa **stop (■)** o reinicia el kernel: significa que el `input()` está esperando o que un proveedor está agotando su timeout.

In [13]:
run_cli()

🤖 Chatbot con fallback (OpenAI → Anthropic → Google)
Escribe '/salir' para terminar.

[Asistente · OpenAI]: 
⚠️  OpenAI no está disponible (APIError: You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.).
➡️  Cambiando a Anthropic…

[Asistente · Anthropic]: 
⚠️  Anthropic no está disponible (BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.'}, 'request_id': 'req_011CbqV6L5nFKoSRA8WNkPrg'}).
➡️  Cambiando a Google Gemini…

[Asistente · Google Gemini]: Soy un modelo lingüístico grande, entrenado por Google.
👋 ¡Hasta luego!


## 10. (Opcional) Prueba rápida de la lógica de fallback sin claves reales

Para comprobar la cascada y la respuesta preconfigurada sin gastar llamadas a las API, puedes usar proveedores simulados que fallan a propósito. Esta celda es solo demostrativa y **no forma parte de la solución entregable**.

In [14]:
class _FakeFail(BaseProvider):
    def __init__(self, name): self.name = name
    def generate(self, conversation):
        raise ConnectionError("simulación de fallo de proveedor")

class _FakeOK(BaseProvider):
    name = "Simulado OK"
    def generate(self, conversation):
        texto = "¡Hola! Soy una respuesta simulada que sí funciona."
        print(texto, end="", flush=True); print()
        return texto

# 1) Cascada hasta un proveedor que funciona
demo = Chatbot([_FakeFail("OpenAI"), _FakeFail("Anthropic"), _FakeOK()],
               system_prompt="demo")
demo.send("Hola, ¿me oyes?")
print("\nHistorial:", demo.conversation.messages)

print("\n" + "-" * 60)

# 2) Todos fallan -> respuesta preconfigurada
demo2 = Chatbot([_FakeFail("OpenAI"), _FakeFail("Anthropic"), _FakeFail("Google")],
                system_prompt="demo")
demo2.send("¿Y ahora?")
print("\nHistorial:", demo2.conversation.messages)


[Asistente · OpenAI]: 
⚠️  OpenAI no está disponible (ConnectionError: simulación de fallo de proveedor).
➡️  Cambiando a Anthropic…

[Asistente · Anthropic]: 
⚠️  Anthropic no está disponible (ConnectionError: simulación de fallo de proveedor).
➡️  Cambiando a Simulado OK…

[Asistente · Simulado OK]: ¡Hola! Soy una respuesta simulada que sí funciona.

Historial: [{'role': 'user', 'content': 'Hola, ¿me oyes?'}, {'role': 'assistant', 'content': '¡Hola! Soy una respuesta simulada que sí funciona.'}]

------------------------------------------------------------

[Asistente · OpenAI]: 
⚠️  OpenAI no está disponible (ConnectionError: simulación de fallo de proveedor).
➡️  Cambiando a Anthropic…

[Asistente · Anthropic]: 
⚠️  Anthropic no está disponible (ConnectionError: simulación de fallo de proveedor).
➡️  Cambiando a Google…

[Asistente · Google]: 
⚠️  Google no está disponible (ConnectionError: simulación de fallo de proveedor).

❌ Ningún proveedor respondió. Usando respuesta preconfi